### PHASE 2 — Tuesday | Geographic Risk Analysis

---

> `Business Request from CEO:`
> "Where are we bleeding money? Which cities have the worst default rates? I need to know which geographic markets are causing this crisis so we can adjust our regional strategies."

> `Tuesday Mindset Unlock:` 
> Yesterday you learned the scale. Today you find where. The most dangerous mistake here is sorting by default rate alone. A city with 25% defaults on 20 loans is not the same problem as a city with 12% defaults on 2,000 loans. Rate tells you frequency. Exposure tells you cost. Only the product of both tells you where to act first. If you sort by rate — you get the wrong halt list. An analyst sorts by impact.

> `CEO'S GEOGRAPHIC AUDIT — "Which cities are destroying our portfolio?"`
> We need to know where defaults are concentrated geographically. The CEO wants a top city halt recommendation — but the decision requires more than a ranked table. A business can't simply "stop lending in Mumbai" without understanding the exposure, the active loan pipeline, and the statistical validity of the signal.

---

Dataset Identification

- Where are we bleeding money geographically?
- Which cities have the worst default rates?
- Which geographic markets are causing the crisis?

> Identified datasets: *loans.csv*, *customers.csv*, *dim_city.csv*, *dim_state.csv*, **

---

In [1]:
## Import dependencies and create Spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 17:14:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
## Load the identified datasets and create a temporary views for SQL queries
loans = spark.read.csv("../datasets/loans.csv", header=True)
customers = spark.read.csv("../datasets/customers.csv", header=True)
dim_city = spark.read.csv("../datasets/dim_city.csv", header=True)
dim_state = spark.read.csv("../datasets/dim_state.csv", header=True)

loans.createOrReplaceTempView("loans")
customers.createOrReplaceTempView("customers")
dim_city.createOrReplaceTempView("dim_city")
dim_state.createOrReplaceTempView("dim_state")

print("Loans dataset:")
loans.show(2)
print("Customers dataset:")
customers.show(2)
print("Dimensional City dataset:")
dim_city.show(2)
print("Dimensional State dataset:")
dim_state.show(2)

Loans dataset:
+-------+-----------+--------------+-----------+-----------+-------------+------------------+----------------+-----------------+-------------+-----------+--------------------+
|loan_id|customer_id|institution_id|loan_amount|loan_status|interest_rate|loan_tenure_months|application_date|disbursement_date|maturity_date| emi_amount|     purpose_of_loan|
+-------+-----------+--------------+-----------+-----------+-------------+------------------+----------------+-----------------+-------------+-----------+--------------------+
|      1|       2440|          2954| 190607.125|  Defaulted|  11.39000034|                84|      08-12-2021|       23-12-2021|   16-11-2028|3302.540039|     Living Expenses|
|      2|       2440|          4741| 425798.375|     Active|  14.43999958|                48|      01-01-2022|       11-01-2022|   21-12-2025|11728.73047|Course Fees + Living|
+-------+-----------+--------------+-----------+-----------+-------------+------------------+------------

STEP 2A — City-Level Loan Volume

> What you're doing: Count total loans per city. This tells you where EduFin is most active geographically. You need this before defaults — because volume determines whether a default signal is meaningful or noise.

*Hint: JOIN loans to dim_city, GROUP BY city_name. COUNT loans per city, ORDER BY total_loans DESC. Decide: INNER JOIN or LEFT JOIN* 

In [3]:
query = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Total Loans`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
ORDER BY `Total Loans` DESC
"""
spark.sql(query).show(10)

+-----------+-----------+
|       City|Total Loans|
+-----------+-----------+
|      Delhi|        209|
|     Raipur|        207|
|     Bhopal|        204|
|Bhubaneswar|        199|
|    Chennai|        191|
|      Patna|        183|
|    Lucknow|        179|
|     Nagpur|        176|
|   Guwahati|        176|
|     Indore|        170|
+-----------+-----------+
only showing top 10 rows


How many cities have fewer than 50 loans? These are candidates for the "insufficient data" exclude list later.


In [4]:
query = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Total Loans`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
HAVING COUNT(*) < 100
"""
spark.sql(query).show(10)

+----+-----------+
|City|Total Loans|
+----+-----------+
+----+-----------+



Query 2A: Geographic Borrower Distribution

- Calculate: Number of customers and loans per city.
- Business Purpose: Identify our largest markets by volume.
- Filter: Focus on disbursed loans only.

In [5]:
query = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Total Loans`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
ORDER BY `Total Loans` DESC
"""
spark.sql(query).show(10)

+-----------+-----------+
|       City|Total Loans|
+-----------+-----------+
|      Delhi|        209|
|     Raipur|        207|
|     Bhopal|        204|
|Bhubaneswar|        199|
|    Chennai|        191|
|      Patna|        183|
|    Lucknow|        179|
|     Nagpur|        176|
|   Guwahati|        176|
|     Indore|        170|
+-----------+-----------+
only showing top 10 rows


STEP 2B — Default Count and Rate Per City

> What you're doing: Calculate total defaults and default rate per city. This is the first time a rate and a volume are in the same row — which means the relationship between them becomes visible.

*Hint: COUNT total, COUNT WHERE status=Defaulted, ROUND(defaulted/total * 100, 2) AS default_rate. GROUP BY city_name.*

In [6]:
query = """
SELECT 
    dc.city_name AS `City`,
    ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END), 2) AS `Defaulted Loans`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*) * 100, 2), '%') AS `Default Rate (%)`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name;
"""
spark.sql(query).show(10)


+-----------+---------------+----------------+
|       City|Defaulted Loans|Default Rate (%)|
+-----------+---------------+----------------+
|  Bangalore|             13|           8.78%|
|      Kochi|             25|          15.15%|
|Bhubaneswar|             19|           9.55%|
|      Jammu|             23|          14.47%|
|      Patna|             17|           9.29%|
|   Vadodara|             11|           7.19%|
|    Gangtok|             17|          11.33%|
|    Chennai|             13|           6.81%|
|    Lucknow|             25|          13.97%|
|     Shimla|             19|          13.19%|
+-----------+---------------+----------------+
only showing top 10 rows


Which city has the highest default RATE? Which has the highest total defaults? Are they the same city? If not — which matters more?

> Highest default rate

In [7]:
query = """
SELECT 
    'Highest Default Rate' AS `Metric`,
    dc.city_name AS `City`,
    SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS `Defaulted Loans`,
    ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS `Default Rate`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) DESC
LIMIT 1
"""
spark.sql(query).show()

+--------------------+----+---------------+------------+
|              Metric|City|Defaulted Loans|Default Rate|
+--------------------+----+---------------+------------+
|Highest Default Rate|Pune|             30|       20.13|
+--------------------+----+---------------+------------+



> Highest total defaults

In [8]:
query = """
SELECT 
    'Highest Total Defaults' AS `Metric`,
    dc.city_name AS `City`,
    SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS `Defaulted Loans`,
    ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS `Default Rate`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) DESC
LIMIT 1
"""

spark.sql(query).show(truncate=False)

+----------------------+------+---------------+------------+
|Metric                |City  |Defaulted Loans|Default Rate|
+----------------------+------+---------------+------------+
|Highest Total Defaults|Ranchi|30             |17.86       |
+----------------------+------+---------------+------------+



- Ranchi: Highest total defaults (30 loans, ₹1.40 Cr) — absolute damage
- Pune: Highest default rate (20.13%) — quality signal
- Why both matter: Rate shows frequency, amount shows cost. Only impact (rate × amount) tells you where to act first.

Query 2B: City-Level Risk Metrics

- Calculate: Default rate per city.
- Business Purpose: Pinpoint which specific locations are underperforming.
- Filter: Exclude cities with low volume (<100 loans) to avoid statistical noise.

In [9]:
query = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Total Loans`,
    SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS `Total Defaults`,
    CONCAT(
        ROUND(
            (SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)), 
            2
        ), 
        '%'
    ) AS `Default Rate (%)`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
HAVING COUNT(*) >= 100
ORDER BY CAST(REPLACE(`Default Rate (%)`, '%', '') AS DOUBLE) DESC
"""
spark.sql(query).show(10)

+---------+-----------+--------------+----------------+
|     City|Total Loans|Total Defaults|Default Rate (%)|
+---------+-----------+--------------+----------------+
|     Pune|        149|            30|          20.13%|
|   Ranchi|        168|            30|          17.86%|
|   Indore|        170|            28|          16.47%|
|   Jaipur|        160|            26|          16.25%|
|   Rajkot|        166|            26|          15.66%|
|    Kochi|        165|            25|          15.15%|
|Ahmedabad|        160|            24|          15.00%|
|    Jammu|        159|            23|          14.47%|
|   Nashik|        135|            19|          14.07%|
|  Lucknow|        179|            25|          13.97%|
+---------+-----------+--------------+----------------+
only showing top 10 rows


STEP 2C — Financial Exposure Per City

> What you're doing: Calculate total defaulted loan value per city in Crores. This is the number that converts "rate" into "cost." A 10% default rate on ₹50 Cr exposure is a ₹5 Cr problem. The same rate on ₹2 Cr exposure is a ₹20L problem. Not the same.

*Hint: SUM(loan_amount) WHERE status=Defaulted, grouped by city. Format as Crores. Also include total disbursed amount per city*

In [10]:
query = """
SELECT 
    dc.city_name AS `City`,
    CONCAT(ROUND(SUM(l.loan_amount) / 10000000.0, 2), ' Cr') AS `Total Disbursed Amount`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN loan_amount * 1.0 ELSE 0 END) / 10000000.0, 2), ' Cr') AS `Total Defaulted Amount`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
ORDER BY ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN loan_amount * 1.0 ELSE 0 END) / 10000000.0, 2) DESC;
"""
spark.sql(query).show(10)

+---------+----------------------+----------------------+
|     City|Total Disbursed Amount|Total Defaulted Amount|
+---------+----------------------+----------------------+
|   Ranchi|               6.71 Cr|                1.4 Cr|
|     Pune|               5.87 Cr|               1.14 Cr|
|  Lucknow|               7.51 Cr|               1.13 Cr|
|   Mumbai|               6.59 Cr|               1.06 Cr|
|Ahmedabad|               7.07 Cr|               1.05 Cr|
|   Indore|               6.73 Cr|               1.04 Cr|
|   Jaipur|               6.58 Cr|               1.03 Cr|
|    Kochi|               6.67 Cr|                1.0 Cr|
|    Jammu|               6.52 Cr|               0.99 Cr|
|   Rajkot|               6.32 Cr|               0.98 Cr|
+---------+----------------------+----------------------+
only showing top 10 rows


Does the city with the highest default RATE also have the highest defaulted AMOUNT? If not — this is the key insight of Phase 2.

- The city with highest default rate is 'Pune' which is 20.13%.
- The city with the highest default amount is 'Ranchi' is 1.4 Cr.

Query 2C: Financial Exposure by Knowledge Hub

- Calculate: Total portfolio value and total defaulted value per city.
- Business Purpose: Identify where the most capital is at risk.
- Formatting: Convert huge sums to Crores/Lakhs.

In [11]:
query = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Total Loans`,
    CONCAT(ROUND(SUM(l.loan_amount) / 10000000.0, 2), ' Cr') AS `Total Portfolio Values`, 
    CASE
        WHEN SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) > 10000000.0 
        THEN CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 10000000.0, 2), ' Cr')
        ELSE CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 100000.0, 2), ' L')
    END AS `Total Defaulted Values` 
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) DESC;
"""
spark.sql(query).show(15)

+---------+-----------+----------------------+----------------------+
|     City|Total Loans|Total Portfolio Values|Total Defaulted Values|
+---------+-----------+----------------------+----------------------+
|   Ranchi|        168|               6.71 Cr|                1.4 Cr|
|     Pune|        149|               5.87 Cr|               1.14 Cr|
|  Lucknow|        179|               7.51 Cr|               1.13 Cr|
|   Mumbai|        168|               6.59 Cr|               1.06 Cr|
|Ahmedabad|        160|               7.07 Cr|               1.05 Cr|
|   Indore|        170|               6.73 Cr|               1.04 Cr|
|   Jaipur|        160|               6.58 Cr|               1.03 Cr|
|    Kochi|        165|               6.67 Cr|                1.0 Cr|
|    Jammu|        159|               6.52 Cr|               98.62 L|
|   Rajkot|        166|               6.32 Cr|               97.93 L|
|   Bhopal|        204|               8.53 Cr|               84.25 L|
|   Nashik|        1

STEP 2D — Geographic Risk Pattern (State-Level Rollup)

> What you're doing: Aggregate by state to find regional patterns. If multiple cities in the same state are defaulting — that suggests a macro factor (economy, employment, institutional quality) rather than a city-specific issue.

*Hint: GROUP BY state_name isntead of city_name. JOIN dim_city to get state. Show: total loans, defaults, default rate, total exposure per state*

In [12]:
query = """
SELECT 
    st.state_name AS `State`,
    st.region AS `Region`,
    COUNT(*) AS `Total Loans`,
    COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) AS `Defaulted Loans`,
    CONCAT(ROUND(COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) * 100.0 / COUNT(*), 2), '%') AS `Default Rate %`,
    CASE
        WHEN SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) >= 10000000.0
        THEN CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) / 10000000.0, 2), ' Cr')
        ELSE CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) / 100000.0, 2), ' L')
    END AS `Total Defaulted Exposure`,
    CASE
        WHEN SUM(CAST(l.loan_amount AS DECIMAL)) >= 10000000.0
        THEN CONCAT(ROUND(SUM(CAST(l.loan_amount AS DECIMAL)) / 10000000.0, 2), ' Cr')
        ELSE CONCAT(ROUND(SUM(CAST(l.loan_amount AS DECIMAL)) / 100000.0, 2), ' L')
    END AS `Total Portfolio Value`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
INNER JOIN dim_state st ON dc.state_id = st.state_id
GROUP BY st.state_name, st.region
ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) DESC;
"""
spark.sql(query).show()

+-----------------+---------+-----------+---------------+--------------+------------------------+---------------------+
|            State|   Region|Total Loans|Defaulted Loans|Default Rate %|Total Defaulted Exposure|Total Portfolio Value|
+-----------------+---------+-----------+---------------+--------------+------------------------+---------------------+
|      Maharashtra|     West|        628|             93|        14.81%|                 3.86 Cr|             25.13 Cr|
|          Gujarat|     West|        479|             61|        12.73%|                 2.53 Cr|             20.04 Cr|
|   Madhya Pradesh|  Central|        374|             50|        13.37%|                 1.88 Cr|             15.26 Cr|
|    Uttar Pradesh|    North|        338|             40|        11.83%|                 1.68 Cr|             13.35 Cr|
|        Jharkhand|     East|        168|             30|        17.86%|                 1.40 Cr|              6.71 Cr|
|        Rajasthan|    North|        160

Which state has the highest total defaulted exposure? What business question does this answer that city-level data cannot?

> Maharashtra with ₹3.86 Crores has the highest total defaulted exposure which also answers the question "Is this a regional crisis?" with a YES. Multiple cities across Maharastra are defaulting at similar rates indicating the fall of an entire state, making the problem MACRO, not MICRO.

Query 2D: Geographic Risk Classification

- Calculate: Classify cities as 'Critical', 'High Risk', 'Monitor', or 'Safe' based on default rates.
- Business Purpose: Prioritize regional intervention teams.
- Rules: defaults > 15% = Critical, > 10% = High Risk.

In [13]:
query = """
SELECT 
    dc.city_name AS `City`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2), '%') AS `Default Rate (%)`,
    CASE
        WHEN ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) > 15 THEN 'Critical'
        WHEN ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) > 10 THEN 'High Risk'
        WHEN ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) > 5 THEN 'Monitor'
        ELSE 'Safe'
    END AS `Risk Category`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
HAVING `Risk Category` IN ('Critical', 'High Risk')
ORDER BY CAST(REPLACE(`Default Rate (%)`, '%', '') AS DOUBLE) DESC;
"""
spark.sql(query).show(10)

+---------+----------------+-------------+
|     City|Default Rate (%)|Risk Category|
+---------+----------------+-------------+
|     Pune|          20.13%|     Critical|
|   Ranchi|          17.86%|     Critical|
|   Indore|          16.47%|     Critical|
|   Jaipur|          16.25%|     Critical|
|   Rajkot|          15.66%|     Critical|
|    Kochi|          15.15%|     Critical|
|Ahmedabad|           15.0%|    High Risk|
|    Jammu|          14.47%|    High Risk|
|   Nashik|          14.07%|    High Risk|
|  Lucknow|          13.97%|    High Risk|
+---------+----------------+-------------+
only showing top 10 rows


STEP 2E — Regional Crisis Dashboard (Final Query)

> What you're doing: Combine city volume, default rate, and financial exposure. Filter to cities with 100+ loans (statistical significance). Sort by financial impact — not default rate. This is your final city-level recommendation output.

*Hint: Full dashboard: city_name, total_loans, default_rate, defaulted_exposure_cr, exposure_rank. HAVING COUNT >= 100. ORDER BY financial exposure DESC. LIMIT top 15* 

In [14]:
query = """
SELECT 
    City,
    State,
    Region,
    `Total Loans`,
    `Defaulted Loans`,
    `Default Rate %`,
    `Defaulted Exposure`,
    `Risk Classification`,
    `Exposure Rank`
FROM (
    SELECT 
        dc.city_name AS `City`,
        st.state_name AS `State`,
        st.region AS `Region`,
        COUNT(*) AS `Total Loans`,
        COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) AS `Defaulted Loans`,
        CONCAT(ROUND(COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) * 100.0 / COUNT(*), 2), '%') AS `Default Rate %`,
        CASE
            WHEN SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) >= 10000000.0
            THEN CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) / 10000000.0, 2), ' Cr')
            ELSE CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) / 100000.0, 2), ' L')
        END AS `Defaulted Exposure`,
        CASE
            WHEN COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) * 100.0 / COUNT(*) >= 15.0
            AND SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) >= 100000000.0
            THEN 'CRITICAL - Halt Immediately'
            WHEN COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) * 100.0 / COUNT(*) >= 14.0
            OR SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) >= 80000000.0
            THEN 'HIGH RISK - Review Urgently'
            WHEN COUNT(CASE WHEN l.loan_status = 'Defaulted' THEN 1 END) * 100.0 / COUNT(*) >= 12.0
            OR SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) >= 50000000.0
            THEN 'MODERATE - Monitor Closely'
            ELSE 'LOW - Continue Operations'
        END AS `Risk Classification`,
        ROW_NUMBER() OVER (ORDER BY SUM(CASE WHEN l.loan_status = 'Defaulted' THEN CAST(l.loan_amount AS DECIMAL) ELSE 0 END) DESC) AS `Exposure Rank`
    FROM loans l
    INNER JOIN customers c ON l.customer_id = c.customer_id
    INNER JOIN dim_city dc ON c.city_id = dc.city_id
    INNER JOIN dim_state st ON dc.state_id = st.state_id
    GROUP BY dc.city_name, st.state_name, st.region
    HAVING COUNT(*) >= 100
)
ORDER BY `Exposure Rank`
LIMIT 15
"""
spark.sql(query).show()

26/04/23 17:14:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 1

+---------+-----------------+-------+-----------+---------------+--------------+------------------+--------------------+-------------+
|     City|            State| Region|Total Loans|Defaulted Loans|Default Rate %|Defaulted Exposure| Risk Classification|Exposure Rank|
+---------+-----------------+-------+-----------+---------------+--------------+------------------+--------------------+-------------+
|   Ranchi|        Jharkhand|   East|        168|             30|        17.86%|           1.40 Cr|HIGH RISK - Revie...|            1|
|     Pune|      Maharashtra|   West|        149|             30|        20.13%|           1.14 Cr|HIGH RISK - Revie...|            2|
|  Lucknow|    Uttar Pradesh|  North|        179|             25|        13.97%|           1.13 Cr|MODERATE - Monito...|            3|
|   Mumbai|      Maharashtra|   West|        168|             22|        13.10%|           1.06 Cr|MODERATE - Monito...|            4|
|Ahmedabad|          Gujarat|   West|        160|      

Name your top 3 halt cities. Now check: if you had sorted by default rate instead of exposure — would you have recommended the same 3?

> The top 3 halt cities are:

- Ranchi
- Pune

Query 2E: Regional Crisis Dashboard

- Calculate: Comprehensive city-level dashboard combining volume, risk, and financial impact.
- Business Purpose: Strategic decision support for regional shutdowns.
- Sorting: Sort by "Total Losses" (Defaulted Value) to prioritize highest financial damage.
- REQUIRED Text Summary: Recommend 3 cities to halt lending in immediately.

In [15]:
query = """
SELECT 
    dc.city_name AS `City`,
    COUNT(*) AS `Loans`,
    CONCAT(ROUND(SUM(l.loan_amount) / 10000000.0, 2), ' Cr') AS `Portfolio Value`,
    CASE
        WHEN SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) > 10000000.0 
        THEN CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 10000000.0, 2), ' Cr')
        ELSE CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 100000.0, 2), ' L')
    END AS `Total Defaulted Values`,
    CONCAT(ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2), '%') AS `Default Rate`,
    ROUND(
        (ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2)) *
        (ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 10000000.0, 2)),
        2
    ) AS `Impact Score`,
    CASE
        WHEN ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) > 15 THEN 'Critical'
        WHEN ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) > 10 THEN 'High Risk'
        WHEN ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) > 5 THEN 'Monitor'
        ELSE 'Safe'
    END AS `Risk Classification`
FROM loans l
INNER JOIN customers c ON l.customer_id = c.customer_id
INNER JOIN dim_city dc ON c.city_id = dc.city_id
GROUP BY dc.city_name
HAVING COUNT(*) >= 100
ORDER BY ROUND(SUM(CASE WHEN l.loan_status = 'Defaulted' THEN l.loan_amount * 1.0 ELSE 0 END) / 10000000.0, 2) DESC;
"""
spark.sql(query).show()

+----------+-----+---------------+----------------------+------------+------------+-------------------+
|      City|Loans|Portfolio Value|Total Defaulted Values|Default Rate|Impact Score|Risk Classification|
+----------+-----+---------------+----------------------+------------+------------+-------------------+
|    Ranchi|  168|        6.71 Cr|                1.4 Cr|      17.86%|        25.0|           Critical|
|      Pune|  149|        5.87 Cr|               1.14 Cr|      20.13%|       22.95|           Critical|
|   Lucknow|  179|        7.51 Cr|               1.13 Cr|      13.97%|       15.79|          High Risk|
|    Mumbai|  168|        6.59 Cr|               1.06 Cr|       13.1%|       13.89|          High Risk|
| Ahmedabad|  160|        7.07 Cr|               1.05 Cr|       15.0%|       15.75|          High Risk|
|    Indore|  170|        6.73 Cr|               1.04 Cr|      16.47%|       17.13|           Critical|
|    Jaipur|  160|        6.58 Cr|               1.03 Cr|      1

> The 3 cities to halt lending it immediately are Ranchi, Pune and Indore due to their high default values and rate reaching upto crtitcal impact.

### BUILD YOUR CUSTOM COLUMNS — Required for Tuesday Submission

> NEW TECHNIQUE — Window Functions: Phase 1 used CASE for classification. Today you unlock RANK(), DENSE_RANK(), and NTILE() — SQL's ranking engine. These functions compute a value ACROSS rows without collapsing them. AI can write a CASE. Only an analyst can choose the right ranking logic.

Column 1 — exposure_rank: Use RANK() OVER (ORDER BY ...) to rank cities by the PRODUCT of default_rate × total_defaulted_exposure. Not rate alone. Not exposure alone. The combined risk signal is the action signal. You must use a window function — not CASE.

Column 2 — risk_quartile: Use NTILE(4) OVER (ORDER BY ...) to divide cities into 4 risk quartiles based on defaulted exposure. Label them: Q1=Critical, Q2=High, Q3=Moderate, Q4=Low. Why NTILE and not CASE? Because NTILE adapts automatically — if 5 new cities enter the portfolio next month, the quartile boundaries shift. Your CASE thresholds from Monday would break.

Your exposure_rank using RANK() OVER (...):

*Hint: RANK() OVER (ORDER BY (default_rate * default_exposure) DESC) AS exposure_rank. Why RANK and not ROW_NUMBER? What happens when cities tie?* 

In [16]:
query = """
WITH city_metrics AS (
    SELECT 
        dc.city_name AS City,
        COUNT(*) AS Loans,
        SUM(l.loan_amount) AS total_portfolio,
        
        -- Defaulted exposure (absolute ₹)
        SUM(CASE 
            WHEN l.loan_status = 'Defaulted' 
            THEN l.loan_amount * 1.0 
            ELSE 0 
        END) AS default_exposure,
        
        -- Default rate (%)
        SUM(CASE 
            WHEN l.loan_status = 'Defaulted' THEN 1 
            ELSE 0 
        END) * 100.0 / COUNT(*) AS default_rate

    FROM loans l
    INNER JOIN customers c ON l.customer_id = c.customer_id
    INNER JOIN dim_city dc ON c.city_id = dc.city_id
    GROUP BY dc.city_name
    HAVING COUNT(*) >= 100
)

SELECT 
    City,
    Loans,
    CONCAT(ROUND(total_portfolio / 10000000.0, 2), ' Cr') AS `Portfolio Value`,
    CASE
        WHEN default_exposure > 10000000 
        THEN CONCAT(ROUND(default_exposure / 10000000.0, 2), ' Cr')
        ELSE CONCAT(ROUND(default_exposure / 100000.0, 2), ' L')
    END AS `Total Defaulted Values`,
    CONCAT(ROUND(default_rate, 2), '%') AS `Default Rate`,
    ROUND(default_rate * (default_exposure / 10000000.0), 2) AS `Impact Score`,
    -- Column 1: Exposure Rank
    RANK() OVER (
        ORDER BY (default_rate * default_exposure) DESC
    ) AS `Exposure Rank`
FROM city_metrics
ORDER BY `Exposure Rank`;
"""
spark.sql(query).show()

26/04/23 17:14:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 1

+----------+-----+---------------+----------------------+------------+------------+-------------+
|      City|Loans|Portfolio Value|Total Defaulted Values|Default Rate|Impact Score|Exposure Rank|
+----------+-----+---------------+----------------------+------------+------------+-------------+
|    Ranchi|  168|        6.71 Cr|                1.4 Cr|      17.86%|       24.97|            1|
|      Pune|  149|        5.87 Cr|               1.14 Cr|      20.13%|       22.96|            2|
|    Indore|  170|        6.73 Cr|               1.04 Cr|      16.47%|       17.07|            3|
|    Jaipur|  160|        6.58 Cr|               1.03 Cr|      16.25%|       16.71|            4|
|   Lucknow|  179|        7.51 Cr|               1.13 Cr|      13.97%|       15.84|            5|
| Ahmedabad|  160|        7.07 Cr|               1.05 Cr|      15.00%|       15.71|            6|
|    Rajkot|  166|        6.32 Cr|               97.93 L|      15.66%|       15.34|            7|
|     Kochi|  165|  

26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Your risk_quartile using NTILE(4) OVER (...):

*Hint: NTILE(4) OVER (ORDER BY defaulted_exposure DESC) AS risk_quartile. Then: CASE risk_quartile WHEN 1 THEN 'Critical' WHEN 2 THEN 'HIGH' WHEN 3 THEN 'MODERATE' ELSE 'LOW' END*


In [17]:
query = """
WITH city_metrics AS (
    SELECT 
        dc.city_name AS City,
        COUNT(*) AS Loans,
        SUM(l.loan_amount) AS total_portfolio,
        
        -- Defaulted exposure (absolute ₹)
        SUM(CASE 
            WHEN l.loan_status = 'Defaulted' 
            THEN l.loan_amount * 1.0 
            ELSE 0 
        END) AS default_exposure,
        
        -- Default rate (%)
        SUM(CASE 
            WHEN l.loan_status = 'Defaulted' THEN 1 
            ELSE 0 
        END) * 100.0 / COUNT(*) AS default_rate

    FROM loans l
    INNER JOIN customers c ON l.customer_id = c.customer_id
    INNER JOIN dim_city dc ON c.city_id = dc.city_id
    GROUP BY dc.city_name
    HAVING COUNT(*) >= 100
)

SELECT 
    City,
    Loans,
    CONCAT(ROUND(total_portfolio / 10000000.0, 2), ' Cr') AS `Portfolio Value`,
    CASE
        WHEN default_exposure > 10000000 
        THEN CONCAT(ROUND(default_exposure / 10000000.0, 2), ' Cr')
        ELSE CONCAT(ROUND(default_exposure / 100000.0, 2), ' L')
    END AS `Total Defaulted Values`,
    CONCAT(ROUND(default_rate, 2), '%') AS `Default Rate`,
    ROUND(default_rate * (default_exposure / 10000000.0), 2) AS `Impact Score`,
    -- Column 1: Exposure Rank
    RANK() OVER (
        ORDER BY (default_rate * default_exposure) DESC
    ) AS `Exposure Rank`,
    -- Column 2: Risk Quartile
    CASE NTILE(4) OVER (ORDER BY default_exposure DESC)
        WHEN 1 THEN 'Critical'
        WHEN 2 THEN 'High'
        WHEN 3 THEN 'Moderate'
        ELSE 'Low'
    END AS `Risk Quartile`
FROM city_metrics
ORDER BY `Impact Score` DESC;
"""
spark.sql(query).show()

26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 1

+----------+-----+---------------+----------------------+------------+------------+-------------+-------------+
|      City|Loans|Portfolio Value|Total Defaulted Values|Default Rate|Impact Score|Exposure Rank|Risk Quartile|
+----------+-----+---------------+----------------------+------------+------------+-------------+-------------+
|    Ranchi|  168|        6.71 Cr|                1.4 Cr|      17.86%|       24.97|            1|     Critical|
|      Pune|  149|        5.87 Cr|               1.14 Cr|      20.13%|       22.96|            2|     Critical|
|    Indore|  170|        6.73 Cr|               1.04 Cr|      16.47%|       17.07|            3|     Critical|
|    Jaipur|  160|        6.58 Cr|               1.03 Cr|      16.25%|       16.71|            4|     Critical|
|   Lucknow|  179|        7.51 Cr|               1.13 Cr|      13.97%|       15.84|            5|     Critical|
| Ahmedabad|  160|        7.07 Cr|               1.05 Cr|      15.00%|       15.71|            6|     Cr

26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 17:14:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Why is NTILE better than hardcoded CASE thresholds here? When would CASE be better? (2–3 sentences):

*Hint: Compare: NTILE adapts to data changes automatically. Your phase 1 CASE used fixed thresholds. When is each approach stronger?*

> NTILE is better than hardcoded CASE thresholds because it automatically splits the cities into equal sized groups (quartiles) and also automatically adapts when new cities are added. Also since our portfolio data can grow or shrink in the near future, the hardcoded thresholds may become outdated making the user update the query time and again. However, in cases where you must stay consistent due to company policies, it is mandatory to use CASE thresholds as they can strictly follow absolute business rules.

### BEFORE YOU SUBMIT — Answer These Three Questions

1. If you sorted your final list by default rate instead of exposure_rank — does your top-3 halt recommendation change? Name the specific cities that move in or out, and explain what that difference means for the CEO's decision.

> The top 3 halt recommendation changes from ('Ranchi', 'Pune', 'Indore') using exposure_rank to ('Pune', 'Ranchi', 'Indore') using default_rank, meaning that no city moved in or out and only the order between 'Ranchi' and 'Pune' has been switched. This difference highlights that default rate prioritizes the frequency of loan failures, making 'Pune' appear riskier due to a higher percentage of defaults. In contrast, exposure-based ranking prioritizes financial impact, where 'Ranchi' ranks higher because it has a larger amount of capital at risk.

2. You excluded cities with fewer than 100 loans. Name one city that was excluded and explain: why does excluding it protect the business from making a bad decision?

> No, city was excluded as there were no cities with fewer than 100 loans.

3. Write your 3-city halt decision memo. Format: City Name → default rate → defaulted exposure → specific reason to halt THIS city and not the next one on the list.

- 'Ranchi' → 17.86% → 1.4 Cr → It is the highest financial exposure in the portfolio (1.40 Cr), combined with one of the highest default rates.
- 'Pune' → 20.13% → 1.14 Cr → It has the highest default rate (20.13%) from our entire list, while also having a decent financial exposure, making it the city with the worst failure frequency.
- 'Indore' → 16.47% → 1.04 Cr → While exposure is slightly lower than Pune and Ranchi, both default rate and absolute loss remain high enough to place it in the critical zone.